In [24]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom

# Customer Behaviour Analysis :


## A reusable Python notebook for studying repeat customer activity

### Why does this analysis matter?

All businesses collect transaction data, but raw rows with data do not immediately explain customer behavior. A useful starting point is to understand whether customers return, how often they purchase, and whether the pattern differs between customer groups.

Repeat customers can be important because they may indicate customer loyalty and ongoing engagement. This analysis help a business to:

- Measure the share of identifiable customers who have made more than one purchase.
- Compare repeat purchase rates across regions or other segments.
- Understand how much evidence supports a comparison.
-Create a simple probability model for a group of customers.

The analysis is descriptive. It can support business questions, but it does not prove why customers return or establish that a marketing action caused a change.

### Research questions

__1. What proportion of customers have made at least a specified number of distinct purchases?__

__2. How does this proportion differ between countries or another available customer group?__

__3. How many repeat customers would be expect in a hypothethical group of customers under a Binomial model?__

__4. How reliable are the comparisons when some groups contain only a small number of customers?__

__5. What data limitations should a business consider before acting on the results?__

For the supplied Online Retail example data, a repeat customer is defined as a customer with at least __two distinct invoices__. This definition can be changed in the configuration section.

### Reuse this notebook with your own data

Yes, that's right! You can download this notebook and run the same type of analysis with your own transaction data, provided the data contains a customer identifier and a way to distinguish purchases or orders.

Only the configuration section above should normally need changing. The rest of the notebook uses those settings.

Minimum required information:

- One column that identifies the customer.
- One column that identifies a distinct order, invoice, purchase, booking, etc.

Optional information:

- A country, region or category column for group comparisons.

- A cancellation indicator, if cancelled transactions should be excluded.

The dataset does not have to use the same column names. For example, you can change CustomerID to ClientID or CustomerNumber in the configuration.



### Dataset used for the example

The example dataset is Online Retail, from the UCI Machine Learning Repository. It contains transactions from a UK-based online retailer between December 2010 and December 2011.

The original dataset has 541 909 rows and eight columns.

Source: Chen, D. (2015), Online Retail [Dataset]. UCI Machine Learning Repository.
https://archive.ics.uci.edu/dataset/352/online+retail

In this dataset:

- CustomerID identifies a customer.
- InvoiceNO identifies an invoice.
- Country identifies the transaction country.
- Invoice numbers begin with __C__ indicate cancellations, according to the dataset documentation.

The Excel file is supplied separately in this project folder. Keep it in the same folder as this notebook, or change `DATA_FILE` below.

### Configuration ❗*Change these values for another dataset* ❗

This is the main section to edit when reusing the notebook.

- `DATA_FILE` - file name or path to your Excel file.
- `CUSTOMER_COL` - customer identifier column.
- `ORDER_COL` - unique order or invoice identifier column.
- `GROUP_COL` - optional for countries, regions, etc. Set to `None` if unvailable.
- `MIN_ORDERS` - number or distinct orders required to count as a repeat customer.
- `CANCEL_PREFIX` - set to `"C"` for the Online Retail invoice convention. Set to `None` if your data does not use this convention.
- `GROUP_ASSIGMENT` - if customers appear in multiple groups, this example assigns each customer to their first recorded group. Choose a rule appropriate to your business.

DO NOT copy the Online Retail cancellation rule to a different dataset unless its documentation confirms that the same rule applies.

In [25]:
# EDIT THIS SECTION FOR YOUR OWN DATA

DATA_FILE = "Online Retail.xlsx"     # e.g. "my_transactions.csv" or "sales.xlsx"

CUSTOMER_COL = "CustomerID"          # e.g. "client_id"
ORDER_COL = "InvoiceNo"              # e.g. "order_id"
GROUP_COL = "Country"                # e.g. "region"; use None if unavailable

MIN_ORDERS = 2                       # 2 means at least two distinct orders
CANCEL_PREFIX = "C"                  # Online Retail only; use None if not applicable

# How to assign one group to each customer if they appear in multiple groups:
# "first" uses the first non-missing value in the file.
GROUP_ASSIGNMENT = "first"

# Binomial scenario: hypothetical number of customers and threshold of interest
SCENARIO_N = 100
SCENARIO_THRESHOLD = 60


### Load the data
The loader supports Excel(.xlsx,xls) and CSV (.csv) files. If your file uses another format, load it with the appropriate pandas function.

The file path is relative to the notebook's working directory unless you provide a full path.

In [26]:
file_path=Path(DATA_FILE)

if not file_path.exists():
    raise FileNotFoundError(f'Could not find `(DATA_FILE)`.')

suffix=file_path.suffix.lower()
if suffix in [".xlsx",".xls"]:
    df=pd.read_excel(file_path)
elif suffix== ".csv":
    df=pd.read_csv(file_path)
else:
    raise ValueError('Supported file types are .xlsx, .xls and .csv.')
print(f'Loaded {len(df):} rows and {df.shape[1]} columns.')
display(df.head())

Loaded 541909 rows and 8 columns.


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Check whether the selected columns exist

This validation makes the notebook easier to reuse. If a column name is incorrect, it gives a clear message before the analysis continues.

In [27]:
required_columns=[CUSTOMER_COL, ORDER_COL]
if GROUP_COL is not None:
    required_columns.append(GROUP_COL)

missing_columns= [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise KeyError(f'These configured columns were not found: {missing_columns}'
                   f'Available columns are: {list(df.columns)}''Update the configuration section')

if MIN_ORDERS <2:
    raise ValueError('MIN_ORDERS should be at least 2 for repeat purchase analysis')

print('Configured columns found:')
print(required_columns)
display(df[required_columns].isna().sum().rename('Missing values').to_frame())

Configured columns found:
['CustomerID', 'InvoiceNo', 'Country']


,Missing values
CustomerID,135080
InvoiceNo,0
Country,0


### Prepare the data

As you can see, we removed rows without a customer ID or order ID because they cannot contribute reliably to a customer-level repeat purchase calculation.

If `CANCEL_PREFIX` is configured, rows whose order ID begins with that prefix are excluded. This rule is appropriate for the supplied UCI Online Retail data. For another dataset, use it only if the source documentation confirms the meaning of the prefix.

In [28]:
df_work=df.copy()
rows_before=len(df_work)

df_work=df_work.dropna(subset=[CUSTOMER_COL, ORDER_COL]).copy()

# Apply a cancellation rule only when one is configured.

if CANCEL_PREFIX is not None:
    order_text=df_work[ORDER_COL].astype(str)
    df_work=df_work.loc[~order_text.str.startswith(CANCEL_PREFIX)].copy()

print(f'Rows in original data: {rows_before:}')
print(f'Rows available for analysis: {len(df_work):}')
print(f'Rows excluded: {rows_before-len(df_work):}')
print(f'Unique customers: {df_work[CUSTOMER_COL].nunique():}')

Rows in original data: 541909
Rows available for analysis: 397924
Rows excluded: 143985
Unique customers: 4339
